In [1]:
import sys
sys.path.append("/kaggle/input/library-metrics")

In [2]:
import numpy as np
import pandas as pd
import psutil, time, threading, tracemalloc
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from metric_toolkit import (
    compute_energy, compute_carbon, compute_edp
)

# ============================================================
# 1️⃣ DỮ LIỆU
# ============================================================
X_train = pd.read_csv('/kaggle/input/data-time-complexity/X_train.csv')
y_train = pd.read_csv('/kaggle/input/data-time-complexity/y_train.csv').values.ravel()

# ============================================================
# 2️⃣ HÀM FEATURE SELECTION: SVM-RFE
# ============================================================
def run_svm_rfe(k=100):
    estimator = SVC(kernel="linear", C=0.1, random_state=42)
    selector = RFE(estimator, n_features_to_select=k, step=0.1)
    selector.fit(X_train, y_train)

    mask = selector.support_
    X_sel = X_train.loc[:, mask]
    num_evals = X_train.shape[1]
    return mask, num_evals, X_sel

# ============================================================
# 3️⃣ LOGGER TRACEMALLOC (Python heap)
# ============================================================
mem_trace_log = []
logging_active = False

def log_tracemalloc(interval=0.1, max_samples=2000):
    start_time = time.time()
    count = 0
    while logging_active and count < max_samples:
        current, peak = tracemalloc.get_traced_memory()
        t = time.time() - start_time
        mem_trace_log.append((t, current / 1024**2, peak / 1024**2))
        time.sleep(interval)
        count += 1

# ============================================================
# 4️⃣ HÀM ĐO THỜI GIAN + BỘ NHỚ (CÓ TIMELINE)
# ============================================================
def measure_time_and_memory_with_timeline(func, *args, **kwargs):
    proc = psutil.Process()
    tracemalloc.start()
    start_time = time.time()

    global logging_active
    logging_active = True
    logger_thread = threading.Thread(target=log_tracemalloc, args=(0.1,))
    logger_thread.start()

    result = func(*args, **kwargs)

    logging_active = False
    logger_thread.join()

    wall_time = time.time() - start_time
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    cpu_util = psutil.cpu_percent(interval=None)
    peak_mem_mb = peak_mem / 1024**2

    return {
        "result": result,
        "WallTime(s)": wall_time,
        "CPUUtil_Avg(%)": cpu_util,
        "PeakMem(MB)": peak_mem_mb,
    }

# ============================================================
# 5️⃣ CHẠY SVM-RFE + GHI LOG
# ============================================================
timing = measure_time_and_memory_with_timeline(run_svm_rfe)
mask, num_evals, X_sel = timing["result"]

wall_time = timing["WallTime(s)"]
cpu_util = timing["CPUUtil_Avg(%)"]
peak_mem = timing["PeakMem(MB)"]
base_mem = psutil.Process().memory_info().rss / 1024**2

# ============================================================
# 6️⃣ TÍNH METRICS NĂNG LƯỢNG
# ============================================================
energy = compute_energy(cpu_util, wall_time)
carbon = compute_carbon(energy)
edp = compute_edp(energy, wall_time)

fs_metrics = {
    "FS_Method": "SVM-RFE",
    "NumFeatures": int(mask.sum()),
    "NumEvals": num_evals,
    "WallTime(s)": wall_time,
    "CPUUtil(%)": cpu_util,
    "PeakMem(MB)": peak_mem,
    "BaseMem(MB)": base_mem,
    "Energy(J)": energy,
    "Carbon(gCO2e)": carbon,
    "EDP(J*s)": edp,
    "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

# ============================================================
# 7️⃣ LƯU FILE KẾT QUẢ
# ============================================================
np.save("/kaggle/working/svmrfe_mask.npy", mask)
pd.DataFrame([fs_metrics]).to_csv("/kaggle/working/fs_metrics_svmrfe.csv", index=False)

mem_df = pd.DataFrame(mem_trace_log, columns=["Time(s)", "Current(MB)", "Peak(MB)"])
mem_df.to_csv("/kaggle/working/tracemalloc_timeline_svmrfe.csv", index=False)

# ============================================================
# 8️⃣ IN RA KẾT QUẢ
# ============================================================
print("✅ SVM-RFE Feature Selection done.")
print(f"→ Selected features: {mask.sum()}")
print(f"→ WallTime: {wall_time:.3f}s | CPU Avg: {cpu_util:.2f}%")
print(f"→ PeakMem (Python heap): {peak_mem:.2f} MB")
print("→ Saved: fs_metrics_svmrfe.csv, tracemalloc_timeline_svmrfe.csv, svmrfe_mask.npy")


✅ SVM-RFE Feature Selection done.
→ Selected features: 100
→ WallTime: 415.302s | CPU Avg: 25.70%
→ PeakMem (Python heap): 374.05 MB
→ Saved: fs_metrics_svmrfe.csv, tracemalloc_timeline_svmrfe.csv, svmrfe_mask.npy
